##  4차 시도: Bbox 기반 최적화 실험 (Detection V4)

### 1. 실험 개요
- **방식**: Object Detection (Bounding Box)
- **목적**: 3차 시도의 Segmentation 방식에서 발생한 소형 결함(Pinhole) 탐지 한계를 극복하고, 대규모 데이터셋(1.2만 장) 환경에서 최적의 성능 도출.
- **데이터 분할**: 8:2 (Train 9,859 / Val 2,465)

### 2. 주요 설정
- **학습률(Learning Rate)**: 0.02 (기존 대비 2배 상향)
- **라벨링**: 숫자 ID 대신 영문 클래스명 적용 (Crack, Pinhole, Corrosion 등)
- **데이터 보강**: Pinhole 데이터를 4,725장을 추가로 확보하여 작은 핀홀(소형 객체)탐지 능력 강화 시도.

### 3. 평가 계획
- **mAP50 성능 확인**: Bbox 회귀를 통한 전반적인 탐지 정확도 향상 여부 검토.
- **시각화 분석**: 원본 이미지와 탐지 결과를 1:1로 비교하여 Pinhole 및 Crack에 대한 오탐지/미탐지 분석.

#### 📊 4차 시도 정량 평가 결과
- **mIoU (Mean IoU)**: 0.8515
- **mAP50 (Box)**: 0.7164
- **주요 개선 사항**: 
  - Segmentation 대비 Bbox 회귀를 통한 추론 속도 향상.
  - Pinhole 및 Crack 등 미세 결함에 대한 IoU 수치 변화 집중 모니터링.

In [3]:
import os
import json
import shutil
import random

# 1. 경로 설정 
base_path = r'C:\Users\SSAFY\Desktop\SafeDeck_Data'
raw_img_dir = os.path.join(base_path, 'images')
raw_json_dir = os.path.join(base_path, 'labels')

# 2. 결과 폴더 생성
dirs = ['images/train', 'images/val', 'labels/train', 'labels/val']
for d in dirs:
    os.makedirs(os.path.join(base_path, d), exist_ok=True)

# 3. 영어 이름 매핑 
ship_map = {
    201: 0, # WaterSpotting
    202: 1, # Sagging
    203: 2, # Peeling
    204: 3, # Pinhole
    205: 4, # Crack
    206: 5, # Blistering
    207: 6, # Inclusion
    301: 7, # WeldingDamage
    302: 8, # Scratch
    303: 9  # Corrosion
}

# 4. 이미지 목록 섞기 (8:2 분할용)
all_images = [f for f in os.listdir(raw_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(all_images)

# 5. 8:2 비율 나누기
split_idx = int(len(all_images) * 0.8)
train_list = all_images[:split_idx]
val_list = all_images[split_idx:]

def process_and_move(file_list, target):
    count = 0
    for img_name in file_list:
        # 1) 이미지 이동
        shutil.move(os.path.join(raw_img_dir, img_name), os.path.join(base_path, 'images', target, img_name))
        
        # 2) JSON -> TXT 변환
        json_name = os.path.splitext(img_name)[0] + '.json'
        src_json = os.path.join(raw_json_dir, json_name)
        dst_txt = os.path.join(base_path, 'labels', target, json_name.replace('.json', '.txt'))
        
        # 3) TXT 파일은 무조건 생성 (정상 이미지의 경우(bbox) 가 비어있는 경우도 학습시키기!)
        with open(dst_txt, 'w') as f_out:
            if os.path.exists(src_json):
                with open(src_json, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                w_img, h_img = data['images'][0]['width'], data['images'][0]['height']
                
                for ann in data['annotations']:
                    if ann['category_id'] in ship_map and 'bbox' in ann:
                        cls_idx = ship_map[ann['category_id']]
                        bx, by, bw, bh = ann['bbox']
                        # YOLO Bbox 형식 (중심x, 중심y, 너비, 높이 - 모두 0~1 사이값)
                        f_out.write(f"{cls_idx} {(bx+bw/2)/w_img:.6f} {(by+bh/2)/h_img:.6f} {bw/w_img:.6f} {bh/h_img:.6f}\n")
        count += 1
    print(f"✅ {target} 데이터 {count}장 처리 완료!")

# 실행
process_and_move(train_list, 'train')
process_and_move(val_list, 'val')

✅ train 데이터 0장 처리 완료!
✅ val 데이터 0장 처리 완료!


# 학습 시작

In [4]:
from ultralytics import YOLO

# 1. 객체 탐지 전용 모델(Bbox 방식) 불러오기
model = YOLO('yolov8n.pt') 

# 2. 학습 시작!
model.train(
    data='ship_data.yaml', 
    epochs=70,        # 학습 횟수를 이전보다 10회 늘림 (60회 -> 70회)
    imgsz=640, 
    batch=16, 
    name='SafeDeck_Detection_V4', 
    device=0,          #  RTX 4070 GPU 가동
    lr0=0.02,          #  학습률을 높여서 더 빠르게 정답을 찾게 설정
    patience=20,       # 20번 동안 성적이 안 오르면 멈추게 설정
    optimizer='AdamW'  # 핀홀 같은 미세 결함을 잡는 데 유리하게 공부 방식 설정함
)

New https://pypi.org/project/ultralytics/8.4.5 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ship_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.02, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=SafeDeck_Detection_V4, nbs=64, n

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([2, 3, 4, 5, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001585EF7D060>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047

# 3. 모델 성능 평가 (Validation)
학습이 끝난 후, 전체 검증 데이터에 대해 지표를 산출하기

In [5]:
# 가장 성능이 좋았던 모델 불러오기
best_model_path = os.path.join(base_path, 'runs', 'detect', 'SafeDeck_Detection_V4', 'weights', 'best.pt')
model = YOLO(best_model_path)

# 검증 수행
metrics = model.val()
print(f"✅ 검증 완료! mAP50: {metrics.box.map50:.4f}")

Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2225.0136.7 MB/s, size: 4032.0 KB)
val: Scanning C:\Users\SSAFY\Desktop\SafeDeck_Data\labels\val.cache... 2345 images, 118 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2345/2345  0.0s
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_02086ce1-623c-40f4-bc1e-821bc7a13506.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_0464ab60-6acc-4fde-a6b4-963e90b2460e.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_06337534-482f-4495-83a2-13e8a2f26723.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_069c901e-7f84-4327-a735-0988b433b3cf.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images

# 4. 랜덤 10장 테스트해보고 저장하기
val 폴더에서 랜덤으로 10장 골라서 탐지 결과 적어보고, outputs 폴더에 저장

In [6]:
import cv2
import matplotlib.pyplot as plt
import random

val_img_dir = os.path.join(base_path, 'images', 'val')
output_dir = os.path.join(base_path, 'outputs_V4')
os.makedirs(output_dir, exist_ok=True)

# 랜덤 10장 선택
val_images = [f for f in os.listdir(val_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample_images = random.sample(val_images, min(10, len(val_images)))

for i, img_name in enumerate(sample_images):
    img_path = os.path.join(val_img_dir, img_name)
    results = model.predict(source=img_path, conf=0.25, device=0)[0]
    
    # 이미지 준비
    orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    labeled_img = cv2.cvtColor(results.plot(), cv2.COLOR_BGR2RGB) # 영어 라벨 포함된 이미지
    
    # 시각화 (원본 vs 결과)
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(orig_img)
    plt.title(f"Original: {img_name}", fontsize=15)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(labeled_img)
    plt.title("SafeDeck AI Detection Result (V4)", fontsize=15)
    plt.axis('off')
    
    plt.tight_layout()
    
    # 결과 저장
    save_path = os.path.join(output_dir, f"v4_result_{i+1}_{img_name}")
    plt.savefig(save_path)
    plt.close()
    print(f"✅ [{i+1}/10] {img_name} 저장 완료")


image 1/1 C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\204_13_d04b529e-3400-49bf-aceb-150421827943.jpg: 480x640 2 Pinholes, 56.0ms
Speed: 6.7ms preprocess, 56.0ms inference, 5.4ms postprocess per image at shape (1, 3, 480, 640)
✅ [1/10] 204_13_d04b529e-3400-49bf-aceb-150421827943.jpg 저장 완료

image 1/1 C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\303_19_1af090a9-8583-4c89-b2e6-34fa7b5da932.jpg: 480x640 1 Corrosion, 44.3ms
Speed: 4.5ms preprocess, 44.3ms inference, 11.7ms postprocess per image at shape (1, 3, 480, 640)
✅ [2/10] 303_19_1af090a9-8583-4c89-b2e6-34fa7b5da932.jpg 저장 완료

image 1/1 C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\204_11_cb2e41f1-4c19-4248-8eaf-4d4d92c0eb49.jpg: 480x640 2 Pinholes, 64.4ms
Speed: 5.3ms preprocess, 64.4ms inference, 12.1ms postprocess per image at shape (1, 3, 480, 640)
✅ [3/10] 204_11_cb2e41f1-4c19-4248-8eaf-4d4d92c0eb49.jpg 저장 완료

image 1/1 C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\204_16_a7435c17-3305-4f03-a561-b6bf9ad117e1.jpg: 640x

In [7]:
import numpy as np
from ultralytics import YOLO
import os

def evaluate_safedeck_v4(model_path, data_yaml):
    # 1. 모델 및 설정 로드
    model = YOLO(model_path)
    
    # 2. 검증 수행 (Confusion Matrix를 얻기 위해 val() 실행)
    print("검증 데이터셋 평가 중...")
    results = model.val(data=data_yaml)
    
    # 3. 혼동 행렬(Confusion Matrix) 추출
    # YOLOv8의 Confusion Matrix는 (정답, 예측) 픽셀 단위가 아닌 '객체 단위'로 계산됩니다.
    cm = results.confusion_matrix.matrix
    
    # 클래스 이름 가져오기 (WaterSpotting, Sagging 등 10개)
    names = model.names
    num_classes = len(names)
    
    iou_per_class = []
    print("\n[클래스별 IoU 상세 분석]")
    
    # 4. 각 클래스별 IoU 계산
    # IoU = 교집합 / 합집합
    for i in range(num_classes):
        intersection = cm[i, i] # 맞게 탐지한 개수 (TP)
        actual_total = cm[i, :num_classes].sum() # 실제 해당 결함의 총 개수
        pred_total = cm[:num_classes, i].sum()   # 모델이 해당 결함이라 예측한 총 개수
        
        union = actual_total + pred_total - intersection
        
        if union > 0:
            iou = intersection / union
            iou_per_class.append(iou)
            print(f" - {names[i]:<15}: {iou:.4f}")
        else:
            iou_per_class.append(np.nan)
            print(f" - {names[i]:<15}: 데이터 없음(Skip)")

    # 5. 최종 mIoU 계산 (산술 평균)
    miou = np.nanmean(iou_per_class)
    
    print("-" * 30)
    print(f"최종 mIoU 성적: {miou:.4f}")
    print(f"Mask mAP50: {results.box.map50:.4f}") # 탐지 정확도도 함께 확인
    print("-" * 30)
    
    return miou, iou_per_class

# 실행하기 
base_path = r'C:\Users\SSAFY\Desktop\SafeDeck_Data'
best_model = os.path.join(base_path, 'runs', 'detect', 'SafeDeck_Detection_V4', 'weights', 'best.pt')
yaml_file = 'ship_data.yaml'

miou_v4, class_ious_v4 = evaluate_safedeck_v4(best_model, yaml_file)

검증 데이터셋 평가 중...
Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1794.8424.3 MB/s, size: 4512.7 KB)
val: Scanning C:\Users\SSAFY\Desktop\SafeDeck_Data\labels\val.cache... 2345 images, 118 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2345/2345  0.0s
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_02086ce1-623c-40f4-bc1e-821bc7a13506.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_0464ab60-6acc-4fde-a6b4-963e90b2460e.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_06337534-482f-4495-83a2-13e8a2f26723.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\SafeDeck_Data\images\val\201_17_069c901e-7f84-4327-a735-0988b433b3cf.jpg: corrupt JPEG restored and saved
val: C:\Users\SSAFY\Desktop\Safe